In [586]:
import pandas as pd
import numpy as np
from pathlib import Path

data = pd.read_csv("../data/sites_data.csv")


In [587]:
rec_risk_limit={
    "high_max": 0.70,
    "med_max": 1,
}
reliab_risk_limit = {
    "high_max": 0.70,
    "med_max": 0.85,
}


In [588]:
data["rec_risk"] = np.select(
    [
        data["rec_perf_ratio"]< rec_risk_limit["high_max"],
        data["rec_perf_ratio"]< rec_risk_limit["med_max"],
    ],
    [
        "High",  #inferior to 0.75% of reference
        "Med",#from 75-<1
    ],
    default="Low"
)
data[["site_id", "rec_perf_ratio", "rec_risk"]].head(5)

,site_id,rec_perf_ratio,rec_risk
0,Site_001,0.71,Med
1,Site_002,0.76,Med
2,Site_003,0.69,High
3,Site_004,0.88,Med
4,Site_005,1.05,Low


In [589]:
data["reliab_risk"] = np.select(
    [
        data["op_reliab"] < reliab_risk_limit["high_max"],
        data["op_reliab"] < reliab_risk_limit["med_max"],
    ],
    [
        "High",
        "Med",
    ],
    default="Low",
)

In [590]:
risk_priority={
    "Low":1,
    "Med": 2,
    "High": 3.
}

In [591]:
#risk classifier
risk_columns = [
    "rec_risk",
    "reliab_risk",
]

high_count = (data[risk_columns] == "High").sum(axis=1)
med_count = data[risk_columns].isin(["Med", "Medium"]).sum(axis=1)

data["risk_level"] = np.select(
    [
        high_count >= 2,
        (high_count == 1) & (med_count == 2),
        (high_count == 1) | (med_count >= 2),
    ],
    [
        "High",
        "High",
        "Medium",
    ],
    default="Low",
)



In [592]:
#controlpoint
data[
    [
        "site_id",
        "rec_risk",
        "reliab_risk",
        "risk_level",
    ]
].head(5)

,site_id,rec_risk,reliab_risk,risk_level
0,Site_001,Med,High,Medium
1,Site_002,Med,High,Medium
2,Site_003,High,High,High
3,Site_004,Med,Med,Medium
4,Site_005,Low,High,Medium


In [593]:
data["risk_level"].value_counts()

risk_level
Medium    247
Low       218
High       35
Name: count, dtype: int64

In [594]:
#validation+
print("Dataset dimensions:", data.shape)
print("Unique site IDs:", data["site_id"].nunique())
print("Missing values:", data.isna().sum().sum())
print("Duplicate site IDs:", data["site_id"].duplicated().sum())

print("\nOverall risk distribution:")
print(data["risk_level"].value_counts())


Dataset dimensions: (500, 10)
Unique site IDs: 500
Missing values: 0
Duplicate site IDs: 0

Overall risk distribution:
risk_level
Medium    247
Low       218
High       35
Name: count, dtype: int64


In [595]:
risk_columns = [ "rec_risk", "reliab_risk"]

for column in risk_columns:
    print(f"\n{column}")
    print(data[column].value_counts())
    print((data[column].value_counts(normalize=True) * 100).round(1))


rec_risk
rec_risk
Low     189
Med     182
High    129
Name: count, dtype: int64
rec_risk
Low     37.8
Med     36.4
High    25.8
Name: proportion, dtype: float64

reliab_risk
reliab_risk
Med     191
Low     191
High    118
Name: count, dtype: int64
reliab_risk
Med     38.2
Low     38.2
High    23.6
Name: proportion, dtype: float64


In [596]:
print(data.groupby("risk_level")[
    ["rec_perf_ratio", "op_reliab"]
].median().round(2))

            rec_perf_ratio  op_reliab
risk_level                           
High                  0.62       0.66
Low                   1.08       0.89
Medium                0.77       0.75


In [597]:
data.to_csv("../data/risks.csv", index=False)